# **Mapa de daño** del parche de loops `p50c` — Qwen3.5-0.8B (Colab L4)

Hasta ahora medíamos el daño de un parche con un solo número (NLL en texto sano). Aquí vemos **dónde** daña:
el mismo texto pasa por el modelo con el parche apagado y prendido (sin generar nada), en 7 tipos de texto:
su propio razonamiento (respuestas sanas guardadas en Drive), mates (GSM8K), prosa (WikiText), código (HumanEval),
chat (UltraChat), español y chino (OPUS-100).

Por dominio: cambio de NLL, KL, % de tokens que cambian mucho (KL > 0.1), % donde la palabra más probable sigue igual,
si la dirección del parche siquiera aparece en ese texto, y los 5 tokens más dañados con su contexto.
Control: un parche al azar del mismo tamaño (mismos módulos y alpha, dirección aleatoria).

~10–15 min en L4. Resultado: `MyDrive/wsi_runs/FINALES/damage_map_qwen35-08b.json`. Al final el runtime se apaga solo.

In [ ]:
MODEL = "Qwen/Qwen3.5-0.8B"
RUN_NAME = "damage_map_qwen35-08b"
OWN_RUN = "discover_blind"     # carpeta en MyDrive/wsi_runs con respuestas sanas del base (dominio own_thinking)
REPO = "/content/wsi"
BASE = "/content/base"
S = f"{REPO}/scripts/experiment_damage_map.py"
PATCH = f"{REPO}/patches/qwen35_0.8b_p50c_loop_fix.moipatch"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q hf_transfer
!pip install -q flash-linear-attention || echo "fla opcional"   # causal-conv1d compiles from source for ages: skip it
import torch, transformers; print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, shutil, json, subprocess, datetime, time
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/wsi_runs"
FINALES = f"{ROOT}/FINALES"
OUTD = f"{ROOT}/{RUN_NAME}"
os.makedirs(FINALES, exist_ok=True); os.makedirs(OUTD, exist_ok=True)
W = "/content/work_own"
src = f"{ROOT}/{OWN_RUN}"
need = ("config.json", "requests_test.jsonl", "outputs_test_base.jsonl")
WORK_ARG = []
if all(os.path.exists(f"{src}/{f}") for f in need):
    os.makedirs(W, exist_ok=True)
    for f in need:
        shutil.copy2(f"{src}/{f}", W)
    WORK_ARG = ["--work", W]
    print("own_thinking <-", src)
else:
    print("sin respuestas guardadas en", src, "-> se salta own_thinking")

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git fetch -q --depth 1 origin main && git reset -q --hard origin/main
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert os.path.exists(S), "el repo no tiene experiment_damage_map.py: falta push"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)

In [ ]:
out = f"{OUTD}/map.json"
if not os.path.exists(out):                       # retoma: lo ya hecho no se repite
    cmd = ["python", "-u", S, "--model", BASE, "--patch", PATCH, "--out", "/content/map.json", "--data-dir", "/content/data"] + WORK_ARG
    subprocess.run(cmd, check=True, cwd=f"{REPO}/scripts")
    shutil.copy2("/content/map.json", out)
RES = json.load(open(out))

In [ ]:
for name, per in RES["map"].items():
    print(f"\n{name}")
    for d, m in per.items():
        print(f"  {d:13s} dNLL {m['dnll']:+.4f} | KL {m['kl']:.4f} | tokens que cambian mucho {m['hit_share']:.1%}"
              f" | misma palabra top-1 {m['top1_agree']:.1%}")
    worst_dom = max(per, key=lambda d: per[d]["kl"]) if per else None
    if worst_dom and not name.endswith("__random"):
        print(f"  tokens más dañados en {worst_dom}:")
        for w in per[worst_dom]["worst"][:3]:
            print(f"    KL {w['kl']:.2f} | ...{w.get('before', '')[-60:]!r} -> real {w.get('actual')!r} | base {w.get('base_top1')!r} vs parche {w.get('patched_top1')!r}")
if RES.get("skipped"):
    print("\ndominios saltados:", RES["skipped"])
final = {"run": RES}
final["_final"] = {"complete": True, "model": MODEL, "commit": COMMIT, "notebook": "damage_map_colab.ipynb",
                   "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
                   "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as fh:
    json.dump(final, fh, indent=1, ensure_ascii=False)
os.replace(dst + ".tmp", dst)
print("\nGuardado final en Drive:", dst)

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    print("Resultado en FINALES. Apagando el runtime en 60 s...", flush=True)
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")